In [19]:
from pyspark.sql import SparkSession
import time

In [2]:
spark = SparkSession.builder \
    .appName("Spark_Passionate") \
    .master("spark://spark-master:7077") \
    .config("spark.ui.port", "4042") \
    .config("spark.executor.instances","2") \
    .config("spark.executor.core","1") \
    .config("spark.executor.memory","512m") \
    .config("spark.cores.max", "2") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/09 11:47:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
sc = spark.sparkContext
rdd = sc.textFile("/opt/spark/work-dir/data/orders.csv",4)

In [37]:
total_lines = rdd.count()

In [38]:
print(total_lines)

2001


In [39]:
first_5 = rdd.take(5)

In [24]:
print(first_5)

['order_id,customer_id,city,product,category,quantity,price', '1,C041,CanTho,Desk,Furniture,4,300', '2,C216,HCM,Phone,Electronics,2,900', '3,C172,Hanoi,Keyboard,Electronics,1,80', '4,C250,HCM,Phone,Electronics,5,900']


In [ ]:
# practice 2 

In [7]:
header = rdd.first()

In [8]:
rdd_no_header = rdd.filter(lambda x: x != header)

In [7]:
total_lines = rdd_no_header.count()

In [7]:
print(total_lines)

2000


In [45]:
first_10 = rdd_no_header.take(10)

In [46]:
print(first_10)

['1,C041,CanTho,Desk,Furniture,4,300', '2,C216,HCM,Phone,Electronics,2,900', '3,C172,Hanoi,Keyboard,Electronics,1,80', '4,C250,HCM,Phone,Electronics,5,900', '5,C215,Hanoi,Chair,Furniture,3,120', '6,C257,Danang,Monitor,Electronics,1,250', '7,C009,Hanoi,Monitor,Electronics,5,250', '8,C261,CanTho,Mouse,Electronics,4,25', '9,C240,Hanoi,Keyboard,Electronics,5,80', '10,C140,Danang,Lamp,Furniture,1,60']


In [ ]:
# practice 3

In [9]:
rdd_split = rdd_no_header.map(lambda row: row.split(','))

In [9]:
row_10 = rdd_split.take(10)

In [10]:
print(row_10)

[['1', 'C041', 'CanTho', 'Desk', 'Furniture', '4', '300'], ['2', 'C216', 'HCM', 'Phone', 'Electronics', '2', '900'], ['3', 'C172', 'Hanoi', 'Keyboard', 'Electronics', '1', '80'], ['4', 'C250', 'HCM', 'Phone', 'Electronics', '5', '900'], ['5', 'C215', 'Hanoi', 'Chair', 'Furniture', '3', '120'], ['6', 'C257', 'Danang', 'Monitor', 'Electronics', '1', '250'], ['7', 'C009', 'Hanoi', 'Monitor', 'Electronics', '5', '250'], ['8', 'C261', 'CanTho', 'Mouse', 'Electronics', '4', '25'], ['9', 'C240', 'Hanoi', 'Keyboard', 'Electronics', '5', '80'], ['10', 'C140', 'Danang', 'Lamp', 'Furniture', '1', '60']]


In [ ]:
# practice 3*

In [54]:
rdd_split_dtype = rdd_split.map(lambda x: (int(x[0]),x[1], x[2], x[3], x[4], int(x[5]), int(x[6])))
                         

In [55]:
rdd_split_dtype.take(5)

[(1, 'C041', 'CanTho', 'Desk', 'Furniture', 4, 300),
 (2, 'C216', 'HCM', 'Phone', 'Electronics', 2, 900),
 (3, 'C172', 'Hanoi', 'Keyboard', 'Electronics', 1, 80),
 (4, 'C250', 'HCM', 'Phone', 'Electronics', 5, 900),
 (5, 'C215', 'Hanoi', 'Chair', 'Furniture', 3, 120)]

In [ ]:
# practice 4

In [56]:
rdd_quantity = rdd_split.map(lambda x: int(x[5]))

In [57]:
print(rdd_quantity.take(5))

[4, 2, 1, 5, 3]


In [59]:
total_quantity = rdd_quantity.reduce(lambda a,b: a + b)

In [60]:
print(total_quantity)

6085


In [ ]:
$ practice 5

In [11]:
rdd_revenue = rdd_split.map(lambda x: int(x[5])*int(x[6])).reduce(lambda a,b: a + b)

In [13]:
print(rdd_revenue)

2293120


In [ ]:
# practice 6

In [10]:
rdd_count_city = rdd_split.map(lambda x: (x[2],1)).reduceByKey(lambda a,b: a + b)

In [11]:
result = rdd_count_city.collect()

In [12]:
print(result)

[('HCM', 498), ('CanTho', 501), ('Hanoi', 490), ('Danang', 511)]


In [14]:
rdd_count_num_category = rdd_split.map(lambda x: (x[4],int(x[5]) * int(x[6])))

In [15]:
print(rdd_count_num_category.take(5))

[('Furniture', 1200), ('Electronics', 1800), ('Electronics', 80), ('Electronics', 4500), ('Furniture', 360)]


In [16]:
result = rdd_count_num_category.reduceByKey(lambda a,b : a + b)

In [17]:
print(result.collect())

[('Electronics', 1975420), ('Furniture', 317700)]


In [ ]:
# practice 8

In [11]:
rdd_split.glom().map(lambda x: x[:3]).collect()

[[['1', 'C041', 'CanTho', 'Desk', 'Furniture', '4', '300'],
  ['2', 'C216', 'HCM', 'Phone', 'Electronics', '2', '900'],
  ['3', 'C172', 'Hanoi', 'Keyboard', 'Electronics', '1', '80']],
 [['510', 'C176', 'Danang', 'Lamp', 'Furniture', '5', '60'],
  ['511', 'C062', 'Hanoi', 'Phone', 'Electronics', '1', '900'],
  ['512', 'C281', 'CanTho', 'Tablet', 'Electronics', '5', '400']],
 [['1015', 'C215', 'Hanoi', 'Lamp', 'Furniture', '5', '60'],
  ['1016', 'C067', 'Danang', 'Desk', 'Furniture', '4', '300'],
  ['1017', 'C083', 'Hanoi', 'Desk', 'Furniture', '2', '300']],
 [['1507', 'C275', 'CanTho', 'Mouse', 'Electronics', '5', '25'],
  ['1508', 'C262', 'CanTho', 'Keyboard', 'Electronics', '1', '80'],
  ['1509', 'C060', 'Danang', 'Monitor', 'Electronics', '3', '250']]]

In [12]:
rdd_split.glom().map(len).collect()

[509, 505, 492, 494]

In [ ]:
# practice 9

In [28]:
rdd_slow = rdd_split.map(lambda x: (time.sleep(0.05),x)[1])

In [29]:
result = rdd_slow.collect()

In [30]:
print(result[:5])

[['1', 'C041', 'CanTho', 'Desk', 'Furniture', '4', '300'], ['2', 'C216', 'HCM', 'Phone', 'Electronics', '2', '900'], ['3', 'C172', 'Hanoi', 'Keyboard', 'Electronics', '1', '80'], ['4', 'C250', 'HCM', 'Phone', 'Electronics', '5', '900'], ['5', 'C215', 'Hanoi', 'Chair', 'Furniture', '3', '120']]


In [31]:
spark.stop()
del spark